# NDT7 (M-Lab) Data Prep — Cambodia Broadband + Mobile, Province x Quarter

Aggregates `data/ndt7/kh/mlab_kh_clean.parquet` (770,501 raw NDT7 test records, already
IP-classified into `category`/`network_type` and province-joined; see
`data/ndt7/kh/README_for_eda.md` and `CLEANING_OVERVIEW.md`) into the same province x quarter
format used for Vietnam (`vietnam_ndt7_prep.ipynb`), split into two parts — Broadband and
Mobile/Cellular — mirroring the combined structure of `notebooks/ookla/cambodia_eda.ipynb`.

Mirrors the Vietnam NDT7 pipeline's methodology: bin raw lat/lon points into Ookla-style
zoom-16 slippy tiles first, then aggregate tiles up to province level, so `n_tiles`/
`is_reliable` stay comparable across Ookla and NDT7 and across countries
(`total_tests >= 100 & n_tiles >= 5`).

**Province name mismatch:** the raw parquet's `province` column uses Khmer-romanized names
with diacritics (e.g. `KâmpóngCham`, `Krâchéh`) that don't match
`data/reference/cambodia_reference.csv` / `data/geo/cambodia_provinces.geojson` (e.g.
`Kampong Cham`, `Kratie`). A manual 25-entry mapping (Section 1.5 below) aligns them before
the province merge.

**Outputs:**
- `data/exports/ndt7_cambodia_province_quarterly.csv` — Broadband
- `data/exports/ndt7_cambodia_mobile_province_quarterly.csv` — Mobile/Cellular

Both use the same column names as `data/exports/ookla_cambodia_province_quarterly.csv`.

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import warnings
warnings.filterwarnings('ignore')

RAW_PARQUET = '../../data/ndt7/kh/mlab_kh_clean.parquet'
KH_REF_CSV  = '../../data/reference/cambodia_reference.csv'

ZOOM = 16
N_TILES = 2 ** ZOOM
MIN_TILE_TESTS = 3

COLS = ['date', 'mean_throughput_mbps', 'min_rtt', 'latitude', 'longitude',
        'type', 'network_type', 'province']

### 1. Province Name Mapping — Raw (Khmer-romanized) → Reference (`province_en`)

In [2]:
# Raw parquet 'province' values (diacritic Khmer romanization) -> cambodia_reference.csv 'province_en'
PROVINCE_MAP = {
    'Batdâmbâng': 'Battambang',
    'BântéayMéanchey': 'Bantey Meanchey',
    'KaôhKong': 'Koh Kong',
    'Kep': 'Kep',
    'KrongPailin': 'Pailin',
    'KrongPreahSihanouk': 'Preah Sihanouk',
    'Krâchéh': 'Kratie',
    'KâmpóngCham': 'Kampong Cham',
    'KâmpóngChhnang': 'Kampong Chhnang',
    'KâmpóngSpœ': 'Kampong Speu',
    'KâmpóngThum': 'Kampong Thom',
    'Kâmpôt': 'Kampot',
    'Kândal': 'Kandal',
    'MôndólKiri': 'Mondulkiri',
    'OtdarMeanChey': 'Oddar Meanchey',
    'PhnomPenh': 'Phnom Penh',
    'Pouthisat': 'Pursat',
    'PreahVihéar': 'Preah Vihear',
    'PreyVêng': 'Prey Veng',
    'Rôtânôkiri': 'Ratanakiri Province',
    'Siemréab': 'Siem Reap',
    'StœngTrêng': 'Stung Treng',
    'SvayRieng': 'Svay Rieng',
    'Takêv': 'Takeo',
    'TbongKhmum': 'Tbong Khmum',
}

ref = pd.read_csv(KH_REF_CSV)
unmapped_targets = set(PROVINCE_MAP.values()) - set(ref['province_en'])
print(f"Mapping covers {len(PROVINCE_MAP)} raw province names -> {len(set(PROVINCE_MAP.values()))} reference provinces")
print(f"Reference has {len(ref)} provinces total")
if unmapped_targets:
    print(f"WARNING — mapped targets not found in reference: {unmapped_targets}")
else:
    print("All mapped targets found in reference.")

Mapping covers 25 raw province names -> 25 reference provinces
Reference has 25 provinces total
All mapped targets found in reference.


### 2. Load & Assign Province — Positive Throughput Only

In [3]:
pf = pq.ParquetFile(RAW_PARQUET)
print(f"Total rows in file: {pf.metadata.num_rows:,}")

chunks = []
for batch in pf.iter_batches(columns=COLS, batch_size=2_000_000):
    df = batch.to_pandas()
    df = df[df['mean_throughput_mbps'] > 0]
    df = df.dropna(subset=['latitude', 'longitude', 'province', 'date'])
    df['province'] = df['province'].map(PROVINCE_MAP)
    df = df.dropna(subset=['province'])
    chunks.append(df)

raw_all = pd.concat(chunks, ignore_index=True)
del chunks
print(f"Rows after province mapping + positive-throughput filter: {len(raw_all):,} ({len(raw_all)/pf.metadata.num_rows:.1%} of total)")
print(raw_all['network_type'].value_counts())
raw_all.head()

Total rows in file: 770,501


Rows after province mapping + positive-throughput filter: 770,501 (100.0% of total)
network_type
broadband    462580
cellular     288780
hosting       19141
Name: count, dtype: int64


,date,mean_throughput_mbps,min_rtt,latitude,longitude,type,network_type,province
0,2025-04-05,44.066986,123.014,11.5583,104.9121,download,cellular,Phnom Penh
1,2024-10-05,2.022731,60.959,11.5583,104.9121,download,cellular,Phnom Penh
2,2025-04-05,44.644418,118.389,11.5583,104.9121,download,cellular,Phnom Penh
3,2024-12-25,33.883917,121.000,11.5583,104.9121,download,cellular,Phnom Penh
4,2025-04-05,47.072185,113.680,11.5583,104.9121,download,cellular,Phnom Penh


### 3. Quarter Labels (Ookla-Style `YYYY-QN`)

In [4]:
raw_all['date'] = pd.to_datetime(raw_all['date'])
raw_all['year_q'] = (
    raw_all['date'].dt.to_period('Q').astype(str)
    .str.replace(r'(\d{4})Q(\d)', r'\1-Q\2', regex=True)
)
raw_all['min_rtt'] = raw_all['min_rtt'].clip(upper=2000)
print(raw_all['year_q'].value_counts().sort_index())

year_q
2023-Q1     40190
2023-Q2     59397
2023-Q3     46511
2023-Q4     46217
2024-Q1     49110
2024-Q2     94333
2024-Q3     69750
2024-Q4     56272
2025-Q1    150604
2025-Q2     68043
2025-Q3     56891
2025-Q4     33183
Name: count, dtype: int64


### 4. Zoom-16 Slippy Tile Assignment

In [5]:
lat_rad = np.radians(raw_all['latitude'].clip(-85.05112878, 85.05112878))
mercator_y = np.log(np.tan(lat_rad) + 1.0 / np.cos(lat_rad))

raw_all['tile_x'] = ((raw_all['longitude'].astype(float) + 180) / 360 * N_TILES).astype(int).clip(0, N_TILES - 1)
raw_all['tile_y'] = ((1 - mercator_y / np.pi) / 2 * N_TILES).astype(int).clip(0, N_TILES - 1)
raw_all['tile_id'] = raw_all['tile_x'].astype(str) + '_' + raw_all['tile_y'].astype(str)
print(f"Assigned tiles for {len(raw_all):,} records, {raw_all['tile_id'].nunique():,} distinct tiles.")

Assigned tiles for 770,501 records, 47 distinct tiles.


### 5. Aggregation Pipeline (shared by Broadband & Mobile)

Same tile → province weighted-aggregation logic as `vietnam_ndt7_prep.ipynb`, wrapped in a
function so it can run once for `network_type == 'broadband'` and once for `'cellular'`.

In [6]:
def build_province_quarterly(raw_all, network_type):
    raw = raw_all[raw_all['network_type'] == network_type].copy()
    print(f"[{network_type}] rows: {len(raw):,}")

    tile_agg = raw.groupby(['year_q', 'tile_id', 'type']).agg(
        tile_mean=('mean_throughput_mbps', 'mean'),
        tile_median=('mean_throughput_mbps', 'median'),
        tile_lat=('min_rtt', 'mean'),
        test_count=('mean_throughput_mbps', 'count'),
        province=('province', lambda s: s.mode().iat[0]),
    ).reset_index()

    tile_agg = tile_agg[tile_agg['test_count'] >= MIN_TILE_TESTS].copy()
    print(f"[{network_type}] tile x quarter x type rows after MIN_TILE_TESTS>={MIN_TILE_TESTS} filter: {len(tile_agg):,}")

    dl = tile_agg[tile_agg['type'] == 'download']
    ul = tile_agg[tile_agg['type'] == 'upload']

    dl_stats = dl.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
        'avg_d_mbps': np.average(g['tile_mean'], weights=g['test_count']),
        'avg_lat_ms_wt': np.average(g['tile_lat'], weights=g['test_count']),
        'total_tests': g['test_count'].sum(),
        'n_tiles': g['tile_id'].nunique(),
    }), include_groups=False).reset_index()

    ul_stats = ul.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
        'avg_u_mbps': np.average(g['tile_mean'], weights=g['test_count']),
    }), include_groups=False).reset_index()

    master = pd.merge(dl_stats, ul_stats, on=['year_q', 'province'], how='outer')
    master = master.rename(columns={'year_q': 'quarter'})
    master['year'] = master['quarter'].str.slice(0, 4).astype(int)
    master['quarter.1'] = master['quarter'].str.slice(6, 7).astype(int)

    master['is_reliable'] = (master['total_tests'] >= 100) & (master['n_tiles'] >= 5)
    print(f"[{network_type}] province x quarter rows: {len(master)} | reliable: {master['is_reliable'].sum()} ({master['is_reliable'].mean():.1%})")

    master = master.merge(
        ref[['province_en', 'region', 'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021',
             'density_per_km2', 'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']],
        left_on='province', right_on='province_en', how='left'
    ).drop(columns=['province_en'])

    missing_ref = master[master['region'].isna()]['province'].unique()
    if len(missing_ref):
        print(f"[{network_type}] WARNING — provinces with no reference match: {list(missing_ref)}")

    return master


EXPORT_COLS = ['province', 'quarter', 'year', 'quarter.1', 'avg_d_mbps', 'avg_u_mbps',
               'avg_lat_ms_wt', 'total_tests', 'n_tiles', 'is_reliable', 'region',
               'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021', 'density_per_km2',
               'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']

---
## Part 1 — Broadband

In [7]:
broadband_master = build_province_quarterly(raw_all, 'broadband')
broadband_master.head()

[broadband] rows: 462,580


[broadband] tile x quarter x type rows after MIN_TILE_TESTS>=3 filter: 313


[broadband] province x quarter rows: 151 | reliable: 0 (0.0%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Bantey Meanchey,5.910550,85.844847,59.0,1.0,11.251138,2023,1,False,Northwest,1,898484,2167.4,135,6225.53,199216.96
1,2023-Q1,Battambang,29.017656,84.042333,6.0,1.0,12.997123,2023,1,False,Northwest,2,1132017,2167.4,97,6225.53,199216.96
2,2023-Q1,Kampong Cham,14.432567,219.129500,8.0,1.0,4.936569,2023,1,False,East,4,1062914,2167.4,234,6225.53,199216.96
3,2023-Q1,Kampong Speu,7.574400,75.383429,14.0,1.0,0.900915,2023,1,False,Southwest,4,924175,2167.4,132,6225.53,199216.96
4,2023-Q1,Kampong Thom,19.127098,104.520349,2645.0,1.0,12.449575,2023,1,False,Center,3,807254,2167.4,58,6225.53,199216.96


In [8]:
out_bb = broadband_master[EXPORT_COLS].copy()
OUT_PATH_BB = '../../data/exports/ndt7_cambodia_province_quarterly.csv'
out_bb.to_csv(OUT_PATH_BB, index=False)
print(f"Exported {len(out_bb)} rows -> {OUT_PATH_BB}")
out_bb.head(3)

Exported 151 rows -> ../../data/exports/ndt7_cambodia_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Bantey Meanchey,2023-Q1,2023,1,5.910550,11.251138,85.844847,59.0,1.0,False,Northwest,1,898484,2167.4,135,6225.53,199216.96
1,Battambang,2023-Q1,2023,1,29.017656,12.997123,84.042333,6.0,1.0,False,Northwest,2,1132017,2167.4,97,6225.53,199216.96
2,Kampong Cham,2023-Q1,2023,1,14.432567,4.936569,219.129500,8.0,1.0,False,East,4,1062914,2167.4,234,6225.53,199216.96


---
## Part 2 — Mobile/Cellular

In [9]:
mobile_master = build_province_quarterly(raw_all, 'cellular')
mobile_master.head()

[cellular] rows: 288,780


[cellular] tile x quarter x type rows after MIN_TILE_TESTS>=3 filter: 93


[cellular] province x quarter rows: 43 | reliable: 0 (0.0%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Kampong Thom,16.942150,123.476180,13594.0,1.0,5.481035,2023,1,False,Center,3,807254,2167.4,58,6225.53,199216.96
1,2023-Q1,Phnom Penh,14.541996,169.373942,2192.0,1.0,6.955191,2023,1,False,Capital,1,2352851,2167.4,3465,6225.53,199216.96
2,2023-Q1,Pursat,12.099882,85.531800,5.0,1.0,6.052613,2023,1,False,West,4,516072,2167.4,41,6225.53,199216.96
3,2023-Q1,Takeo,15.466607,106.679075,40.0,1.0,8.219753,2023,1,False,South,1,1097243,2167.4,308,6225.53,199216.96
4,2023-Q2,Kampong Thom,11.680324,105.550784,31137.0,1.0,7.630777,2023,2,False,Center,3,807254,2167.4,58,6225.53,199216.96


In [10]:
out_mb = mobile_master[EXPORT_COLS].copy()
OUT_PATH_MB = '../../data/exports/ndt7_cambodia_mobile_province_quarterly.csv'
out_mb.to_csv(OUT_PATH_MB, index=False)
print(f"Exported {len(out_mb)} rows -> {OUT_PATH_MB}")
out_mb.head(3)

Exported 43 rows -> ../../data/exports/ndt7_cambodia_mobile_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Kampong Thom,2023-Q1,2023,1,16.942150,5.481035,123.476180,13594.0,1.0,False,Center,3,807254,2167.4,58,6225.53,199216.96
1,Phnom Penh,2023-Q1,2023,1,14.541996,6.955191,169.373942,2192.0,1.0,False,Capital,1,2352851,2167.4,3465,6225.53,199216.96
2,Pursat,2023-Q1,2023,1,12.099882,6.052613,85.531800,5.0,1.0,False,West,4,516072,2167.4,41,6225.53,199216.96


## Summary

- Input: 770,501 raw NDT7 test records for Cambodia (2023–2025), IP-classified into
  Broadband/Mobile/Hosting; Hosting excluded (server/CDN traffic, not real users)
- 25-entry manual province-name mapping applied (raw parquet uses diacritic Khmer
  romanization; reference/geojson use plain English names)
- Output: province x quarter aggregates for Broadband and Mobile separately, tile-binned at
  Ookla's zoom-16 resolution, same `is_reliable` threshold as every Ookla country notebook